## Exercise 4
Using the solutions from the previous exercises, write a script that retrieves the data from all categories and saves it into a csv file named `coderslab-shop-data.csv`.

The file should be a table with the following headings:
```
name | price | description_short | qty | category
```

Set `|` as the column separator 

### Hint
- if you want to track the progress of retrieving category data you can use the `tqdm` library (not discussed in class - [click](https://pypi.org/project/tqdm/)),
- change the default file encoding to write the currency symbol correctly (unless it has been removed earlier),
- it may be useful to set the appropriate new line character when opening the file.

In [2]:
# import required libraries here
import requests
import csv

from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
# copy-paste the previous functions definitions here

def get_categories_urls():
    r = requests.get("https://prod-kurs.coderslab.pl/index.php")
    soup = BeautifulSoup(r.text, 'html.parser')   # parsiramo informacije sa stranice

    soups = (
        soup
        .find(attrs={'id': 'top-menu'}) # trazime atribute sa id "top-menu" trazeni elementi su u navbaru
        .find_all('a', attrs={'data-depth': '0'}) # izvlacimo samo linkove i ne gledamo sta je ugnjezdeno dalje
    )
    print(soups)
     

    categories_url = [
        {
            'url': x.attrs['href'], # pravimo dict u listi
            'name': x.text.replace('\ue313\n\ue316\n\n\n', '').strip() # pravimo dict u listi i uklanjamo prazna polja slova i razmake

        }
        for x in soups # petja koja prolazi kroz soups i puni listu 
    ]
    return categories_url
    

def download_category_items(category_url):
    html = requests.get(category_url)
    soup = BeautifulSoup(html.text, 'html.parser')

    items = (
        soup
        .find(id='products')
        .find_all('article')
    )
    
    urls = [x.find('a').attrs["href"] for x in items]

    return urls

def download_product_data(product_url):
    r = requests.get(product_url)
    soup = BeautifulSoup(r.text, 'html.parser')

    data = (
        soup
        .find(id='main')
        .find_all('div', class_='col-md-6')[1]
    )

    name = data.find('h1').text
    price = data.find('span', class_='current-price-value').text.strip()
    description_short = data.find(class_='product-description').text.strip()

    try:
        qty = (
            data
            .find('div', class_='product-quantities')
            .text
            .replace('Items', '')
            .replace('In stock', '')
            .strip()
        )
    except AttributeError:
        qty = 0

    result = {
        'name': name,
        'price': price,
        'description_short': description_short,
        'qty': qty
    }
    return result

In [4]:
# get the list of available categories here
categories = get_categories_urls()

[<a class="dropdown-item" data-depth="0" href="https://mystore-testlab.coderslab.pl/index.php?id_category=3&amp;controller=category">
<span class="float-xs-right hidden-md-up">
<span class="navbar-toggler collapse-icons" data-target="#top_sub_menu_54806" data-toggle="collapse">
<i class="material-icons add"></i>
<i class="material-icons remove"></i>
</span>
</span>
                                Clothes
              </a>, <a class="dropdown-item" data-depth="0" href="https://mystore-testlab.coderslab.pl/index.php?id_category=6&amp;controller=category">
<span class="float-xs-right hidden-md-up">
<span class="navbar-toggler collapse-icons" data-target="#top_sub_menu_49188" data-toggle="collapse">
<i class="material-icons add"></i>
<i class="material-icons remove"></i>
</span>
</span>
                                Accessories
              </a>, <a class="dropdown-item" data-depth="0" href="https://mystore-testlab.coderslab.pl/index.php?id_category=9&amp;controller=category">
    

In [ ]:
# get product information here
records = []  # Prazna lista za sve proizvode

# VANJSKA PETLJA - ide kroz sve kategorije sa progres barom
for category in tqdm(categories):
    # Raspakovaje vrijednosti: URL i naziv kategorije iz dicta
    category_url, category_name = category.values()

    # Preuzima sve URL-ove proizvoda u toj kategoriji
    items = download_category_items(category_url)

    # UNUTRAŠNJA PETLJA - ide kroz sve proizvode u toj kategoriji
    for item in items:
        # Preuzima podatke za taj proizvod (naziv, cijena, opis, količina)
        record = download_product_data(item)
        
        # Dodaje naziv kategorije u record dictionary
        record['category'] = category_name
        
        # Dodaje kompletan record (sa svim podacima i kategorijom) u master listu
        records.append(record)


100%|██████████| 3/3 [00:14<00:00,  4.82s/it]


In [6]:
headers = [
    "name",
    "price",
    "description_short",
    "qty",
    "category"
]

In [7]:
# save results to CSV file here
with open(
        'coderslab-shop-data.csv', 
        'w', 
        newline='', 
        encoding='UTF-8'
    ) as f:
    
    csv_file = csv.writer(
        f, 
        delimiter='|')
    csv_file.writerow(headers)

    for record in records:
        csv_file.writerow(record.values())